# Hafta 4 — Ön İşleme Pipeline'ı ve İlk Model

Ön işleme `src/preprocessing.py`, CV düzeni ve modeller `src/modeling.py`, özet tablo `src/evaluate.py` içindedir. Rapordaki sayıların kaynağı `notebooks/build_ilk_model.py` ile üretilen `hafta4_ozet.json` dosyasıdır. Test seti ayrılır ama hiçbir hücrede kullanılmaz.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_raw
from src.evaluate import summarize_cv
from src.model_plots import plot_pr_roc_curves
from src.modeling import build_baseline, build_logistic_regression, cross_validate_model, make_cv
from src.preprocessing import build_linear_feature_pipeline, split_train_test

split = split_train_test(load_raw())
print('eğitim:', split.X_train.shape, 'test:', split.X_test.shape)

## 1. Ön işleme çıktısı

In [ ]:
features = build_linear_feature_pipeline().fit(split.X_train)
names = features.named_steps['preprocess'].get_feature_names_out()
print(len(names), 'özellik')
list(names)

## 2. 5 katlı katmanlı CV: referans model ve Lojistik Regresyon

In [ ]:
cv = make_cv()
baseline = cross_validate_model(build_baseline(), split.X_train, split.y_train, cv)
logistic = cross_validate_model(build_logistic_regression(), split.X_train, split.y_train, cv)
pd.concat({'Referans': summarize_cv(baseline.fold_metrics), 'Lojistik Regresyon': summarize_cv(logistic.fold_metrics)}, axis=1).round(4)

In [ ]:
plot_pr_roc_curves(split.y_train, logistic.oof_proba, logistic.fold_ids, 'Lojistik Regresyon');